In [ ]:
# Side-by-side: image grid + embeddings colored by region
samples_panel    = fo.Panel(type="Samples", pinned=True)
embeddings_panel = fo.Panel(
    type="Embeddings",
    state=dict(
        brainKey="full_emb_tsne_dinov3",
        colorBy="region"   # visually separate NC vs WP
    )
)

space = fo.Space(
    children=[
        fo.Space(children=[samples_panel]),
        fo.Space(children=[embeddings_panel]),
    ],
    orientation="horizontal"
)

dataset.save_workspace("domain_analysis", space)

# Organize SideBar Groups

In [ ]:
import fiftyone as fo

all_fields = set(dataset.get_field_schema().keys())

# ── Field lists ───────────────────────────────────────────────────────────────
seed0_fields     = [f for f in all_fields if "seed0" in f]
seed0_nms_fields = [f for f in seed0_fields if f.endswith("_nms")]
seed0_raw_fields = [f for f in seed0_fields if f.endswith("_raw")]

core_fields = [
    "filepath", "tags", "metadata", "region", "mission_name",
    "parent_name", "ground_truth", "m_flight", "name_plot",
    "cluster_label", "cluster_label_8",
    "uniqueness_score", "uniqueness_score_per_cluster",
    "representativeness_score", "soft_coverage_score",
]

embeddings_fields = [
    "full_embeddings",
    "full_emb_umap_dinov3",
    "v2_full_emb_umap_dinov3",
]

_EVAL_SUFFIXES = ("_tp", "_fp", "_fn", "_tn", "_iou", "_eval")

eval_fields = [
    f for f in all_fields
    if any(f.endswith(s) for s in _EVAL_SUFFIXES)
]

assigned = set(core_fields + seed0_fields + embeddings_fields + eval_fields)

other_fields = [
    f for f in all_fields
    if f not in assigned
]

# ── Validate — drop any path not in schema ────────────────────────────────────
def _validate(name, paths):
    missing = [p for p in paths if p not in all_fields]
    valid   = [p for p in paths if p in all_fields]
    if missing:
        print(f"  WARN '{name}' — dropped {len(missing)} missing: {missing}")
    print(f"  '{name}' — {len(valid)} fields")
    return valid

core_fields       = _validate("core",       core_fields)
seed0_nms_fields  = _validate("seed0_nms",  seed0_nms_fields)
seed0_raw_fields  = _validate("seed0_raw",  seed0_raw_fields)
embeddings_fields = _validate("embeddings", embeddings_fields)
eval_fields       = _validate("eval",       eval_fields)
other_fields      = _validate("other",      other_fields)

# ── Apply ─────────────────────────────────────────────────────────────────────
dataset.app_config.sidebar_groups = [
    fo.SidebarGroupDocument(
        name="core",       paths=core_fields,       expanded=True),
    fo.SidebarGroupDocument(
        name="seed0_nms",  paths=seed0_nms_fields,  expanded=True),
    fo.SidebarGroupDocument(
        name="seed0_raw",  paths=seed0_raw_fields,  expanded=False),
    fo.SidebarGroupDocument(
        name="embeddings", paths=embeddings_fields, expanded=False),
    fo.SidebarGroupDocument(
        name="eval",       paths=eval_fields,       expanded=False),
    fo.SidebarGroupDocument(
        name="other",      paths=other_fields,      expanded=False),
]

dataset.save()

# ── Reload the App to pick up the new config ──────────────────────────────────
try:
    session.refresh()
    print("Session refreshed.")
except NameError:
    print("No active session — config will apply on next launch_app().")

In [ ]:
all_fields = dataset.get_field_schema().keys()
seed0_fields = [f for f in all_fields if "_SEED0_" in f]
core_fields = [
    'id', 'filepath', 'tags', 'metadata', 'created_at', 
    'last_modified_at', 'region',
    'subregion', 'mission_name',  'ground_truth', 
    'uniqueness', 'flight_plan', 'audit', 
    'stratify_key', 'roi_grid', 'new_stratify_key', 'stem_filepath'
]
embeddings_fields = [ 'full_embeddings', 'full_emb_tsne_dinov3'] 
ecological_fields = [
    'sea_state', 'turbidity_global', 'turbidity_local', 
    'sun_glitter', 'cloud_reflection', 'habitat_type', 'background_complexity', 
    'coral', 'sand', 'dense_seagrass', 'open_sea', 'sparse_seagrass'
]

# Everything else goes into a collapsed "other" group
assigned = set(core_fields + seed0_fields + embeddings_fields + ecological_fields)
other_fields = [f for f in all_fields if f not in assigned]
other_fields

In [ ]:
dataset.app_config.sidebar_groups = [
    fo.SidebarGroupDocument(name="core",    paths=core_fields,   expanded=True),
    fo.SidebarGroupDocument(name="seed_0",  paths=seed0_fields,  expanded=True),
    fo.SidebarGroupDocument(name="ecological", paths=ecological_fields,  expanded=False),
    fo.SidebarGroupDocument(name="embeddings", paths=embeddings_fields,  expanded=False),
    fo.SidebarGroupDocument(name="other",   paths=other_fields,   expanded=False),
]
dataset.save()

In [ ]:
which_seed = "_SEED0_"
all_fields = dataset.get_field_schema().keys()
seed0_fields = [f for f in all_fields if which_seed in f]

seed0_fields_raw = [f for f in seed0_fields if "raw" in f]
seed0_fields_nms = [f for f in seed0_fields if "nms" in f]
seed0_fields_excluding_evaluation = [
    f for f in seed0_fields if not any(x in f for x in ['tp', 'tn', 'fp','fn'])
]
seed0_fields_excluding_evaluation

## downsample tumbnail

In [ ]:
import fiftyone as fo
from PIL import Image
from pathlib import Path

# Generate thumbnails once
thumb_dir = Path("/share/home/e2406743/dataset/exported_img/thumbnails_flplan/")
thumb_dir.mkdir(exist_ok=True)

for sample in dataset.iter_samples(autosave=True, progress=True):
    src  = Path(sample.filepath)
    dst  = thumb_dir / src.name
    if not dst.exists():
        img = Image.open(src)
        img.thumbnail((800, 800), Image.LANCZOS)
        img.save(dst, "JPEG", quality=75)
    sample["thumbnail_path"] = str(dst)

# Tell the App to use thumbnails in the grid
dataset.app_config.media_fields   = ["filepath", "thumbnail_path"]
dataset.app_config.grid_media_field = "thumbnail_path"
dataset.save()

# Indexing in the app

In [ ]:
# Quick readable summary
info = dataset.get_index_information()
print(f"\n{len(info)} indexes on '{dataset.name}':")
for name in sorted(info.keys()):
    print(f"  {name}")

# ── Single field indexes — fields you filter most in the sidebar ──────────────
single_indexes = [
    "tags",                          # match_tags — used constantly
    "m_flight",                      # flight mission filter
    "region",                        # NC / WP / FLPLAN filter
    "cluster_label",                 # cluster browsing
    "cluster_label_8",               # second clustering
    "ground_truth.detections.label", # dugong label filter
    "uniqueness_score",              # score range slider
    "uniqueness_score_per_cluster",  # score range slider
    "soft_coverage_score",           # score range slider
    "representativeness_score",      # score range slider
    "mission_name",                  # mission filter
    "parent_name",                   # parent image filter
]

for field in single_indexes:
    try:
        dataset.create_index(field)
        print(f"  OK  {field}")
    except Exception as e:
        print(f"  SKIP {field} — {e}")


# ── Compound indexes — combinations you filter together most often ────────────
# Rule: list fields in the order you apply them in the sidebar,
# most selective filter first.
compound_indexes = [
    # browsing a specific cluster within a flight mission
    [("cluster_label", 1),   ("m_flight", 1)],

    # browsing predictions by tag (test split) + cluster
    [("tags", 1),            ("cluster_label", 1)],

    # browsing ground truth by tag + label
    [("tags", 1),            ("ground_truth.detections.label", 1)],

    # uniqueness exploration within cluster
    [("cluster_label", 1),   ("uniqueness_score_per_cluster", 1)],

    # coverage score within cluster
    [("cluster_label", 1),   ("soft_coverage_score", 1)],

    # region + flight — common in multi-domain datasets
    [("region", 1),          ("m_flight", 1)],
]

for fields in compound_indexes:
    try:
        dataset.create_index(fields)
        names = [f for f, _ in fields]
        print(f"  OK  compound {names}")
    except Exception as e:
        names = [f for f, _ in fields]
        print(f"  SKIP compound {names} — {e}")


# ── Verify what was created ───────────────────────────────────────────────────
print("\nActive indexes:")
for name, info in dataset.get_index_information().items():
    print(f"  {name:<55}  unique={info.get('unique', False)}")

dataset.save()

import fiftyone as fo
from PIL import Image
from pathlib import Path

# Generate thumbnails once
thumb_dir = Path("/share/home/e2406743/dataset/exported_img/thumbnails_flplan/")
thumb_dir.mkdir(exist_ok=True)

for sample in dataset.iter_samples(autosave=True, progress=True):
    src  = Path(sample.filepath)
    dst  = thumb_dir / src.name
    if not dst.exists():
        img = Image.open(src)
        img.thumbnail((800, 800), Image.LANCZOS)
        img.save(dst, "JPEG", quality=75)
    sample["thumbnail_path"] = str(dst)

# Tell the App to use thumbnails in the grid
dataset.app_config.media_fields   = ["filepath", "thumbnail_path"]
dataset.app_config.grid_media_field = "thumbnail_path"
dataset.save()